# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets (`@id`), their fields, and columns.

In [ ]:
# List available record sets and their fields by `@id` for reference
# Access via metadata for reproducibility using @id only

record_sets = []
if hasattr(md, 'record_sets'):
    # Modern mlcroissant
    record_sets = md.record_sets
elif hasattr(md, 'recordSet'):
    # Legacy field name
    record_sets = md.recordSet

if not record_sets or len(record_sets) == 0:
    # If not present, try loading one by inspecting the available data objects
    print("No record sets found in metadata. Attempting to extract record set info from files...")
    # The record_sets field may be empty in this Croissant export; fallback will be used below.
else:
    for rs in record_sets:
        print(f"Record set @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', str(rs))}")
        # List fields for this record set
        if isinstance(rs, dict):
            fields = rs.get('field', [])
        else:
            fields = getattr(rs, 'field', [])
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            elif hasattr(f, '@id'):
                print(f"    - {f.@id}")
            else:
                print(f"    - {str(f)}")
        print("")

# If record sets (RecordSet) are not exposed via metadata, list by scanning the Croissant schema
croissant_dict = md.to_json()
record_set_ids = []
if 'distribution' in croissant_dict:
    for dist in croissant_dict['distribution']:
        # Each distribution may reference files or record sets
        # Not explored here but can be used for advanced referencing
        pass

# Let us enumerate the record sets available via the python API as per mlcroissant 1.0+
available_record_sets = list(dataset.iter_record_sets())
if not available_record_sets:
    print("No record sets available via python API.")
else:
    for rs in available_record_sets:
        print(f"Record Set @id: {rs['@id']} - Name: {rs['name']}")
        print("  Fields (by @id):")
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"    - {field['@id']} (name: {field.get('name','')})")
            else:
                print(f"    - {field}")
        print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, the Croissant schema exposes record sets via the API.
# We'll enumerate them and extract their @id for extraction.

record_set_info = list(dataset.iter_record_sets())
`#` If the Record Set structure is empty via the schema, fallback to autodiscover from url (mlcroissant auto finds them)
if not record_set_info:
    print("No record sets discovered. Cannot proceed to extraction.")
else:
    record_set_ids = [x['@id'] for x in record_set_info]
    print(f"Record sets available: {record_set_ids}")

    dataframes = {}
    for record_set_id in record_set_ids:
        # Print the first 2 records
        print(f"\nLoading records for RecordSet @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print("  No records found.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))

    # For this notebook, select the first record set for further analysis
    if dataframes:
        main_record_set_id = record_set_ids[0]
        print(f"\nProceeding with Record Set: {main_record_set_id}")
        print("Columns available:")
        print(list(dataframes[main_record_set_id].columns))
    else:
        main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, and grouping data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is None:
    print("No valid record set available for EDA.")
else:
    df = dataframes[main_record_set_id]
    # Display summary statistics
    print("DataFrame summary statistics:")
    print(df.describe(include='all'))

    # Select a numeric field (column) for EDA
    # We'll choose 'MW' or 'molecular_weight' (common for mass spec), otherwise pick the first numeric column.
    candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if 'MW' in df.columns:
        numeric_field = 'MW'
    elif 'molecular_weight' in df.columns:
        numeric_field = 'molecular_weight'
    elif candidates:
        numeric_field = candidates[0]
    else:
        print("No numeric field found for EDA.")
        numeric_field = None

    if numeric_field is not None:
        threshold = df[numeric_field].mean() + df[numeric_field].std() # One std above mean, example
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Pick a group field if present (such as protein_category, accession, or sample)
        group_fields = [col for col in df.columns if col not in candidates and df[col].dtype == object]
        group_field = None
        for g in ['accession', 'sample', 'category']:
            if g in df.columns:
                group_field = g
                break
        if not group_field and group_fields:
            group_field = group_fields[0]  # First available string column

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (showing mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric field for further analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
if main_record_set_id is None or numeric_field is None:
    print("No data to visualize.")
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        # Boxplot grouped by group_field
        plt.figure(figsize=(10, 5))
        order = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).index[:10]
        sns.boxplot(data=df, x=group_field, y=numeric_field, order=order)
        plt.title(f'{numeric_field} by {group_field} (top categories)')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR² dataset on mass spectrometry analysis of extracellular vesicles from the Croissant schema.

- We explored its record sets, fields, and extracted tabular data using the entities' `@id` as required.
- We performed exploratory data analysis on the main numeric attribute (`MW` or molecular weight), filtering and normalizing high-value entries and grouped them by relevant attributes such as accession or sample if available.
- Finally, we visualized the distribution of molecular weights and their grouping, aiding biomarker discovery and other downstream analyses.

This dataset is ready for use in downstream protein expression analysis and benchmarking of bioinformatics workflows using reproducible, machine-actionable metadata.